# Testing Tool Calls

The first test checked the *shape* of the answer. This one checks *behavior*: which tools the agent used, and in what order. Run top to bottom. The lab comes first; the **Appendix** holds tiny runnable examples for the ideas that are new here. Open one only when a line confuses you.

```text
run_agent_stream(agent, "llm as a judge")
        ↓
result.new_messages()          the recording: prompt, tool calls, tool returns, answer
        ↓
collect_tools(messages)        keep only real tool calls, in order
        ↓
assert  first is "search", then "get_file"
```

Think of a flight recorder: the test does not watch the agent live, it reads the recording afterward.

**New terms:** a **message** is one item in the recording, and it holds **parts**; `part_kind` says what a part is (`user-prompt`, `tool-call`, `tool-return`, `text`). `final_result` is PydanticAI's *structured-output* tool: the model "calls" it to hand back the answer object, so it is not a documentation lookup.

**Already covered, so not repeated:** dataclasses and decorators (modularization lesson), `async`/`await` and the event loop, `sys.path` and `pythonpath` (first agent test notebook).

**Differences from the video (all tested):**

| Video | Here | Why |
|---|---|---|
| "No need to parse JSON" for `args` | `args` is a JSON **string**; use `json.loads` for values | Checked on real framework objects and the real model's printed output (Appendix A1) |
| `tests/__init__.py` required | Created, but not strictly needed | Collection still worked with it removed, because `pythonpath = ["."]` makes `tests.utils` importable |
| Real-model test only | A free fake-model test first | Checks the logic with no cost; it proves wiring, not real model behavior |

## 1. Setup

Finds the project, moves into it (so `.cache` is the project's), and stops with a clear message if the kernel is wrong. **Select the kernel that uses `documentation-agent/.venv`.** This notebook never edits your files except to create missing small ones in section 5.

In [ ]:
import importlib.util
import os
import sys
from pathlib import Path

# Show which Python runs this notebook.
print("Notebook Python:", sys.executable)


# Find the project folder (the one holding doc_agent.py and tools.py) from here upward.
def find_project_dir():
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "doc_agent.py").exists() and (folder / "tools.py").exists():
            return folder
        child = folder / "documentation-agent"
        if (child / "doc_agent.py").exists() and (child / "tools.py").exists():
            return child
    raise FileNotFoundError("Could not find doc_agent.py and tools.py. Open this notebook inside documentation-agent/tests/.")


PROJECT_DIR = find_project_dir()
# Work from the project folder and make "import tools" / "import tests.utils" work.
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
print("Project directory:", PROJECT_DIR)

# Stop with one clear message if this kernel lacks a package the cells import.
missing = [p for p in ("pydantic_ai", "pytest", "minsearch", "gitsource", "jaxn", "dotenv") if importlib.util.find_spec(p) is None]
if missing:
    raise RuntimeError(f"This kernel cannot import {missing}. Select {PROJECT_DIR / '.venv' / 'bin' / 'python'} as the kernel.")

# doc_agent.py calls AgentStream.stream_responses(), which exists in pydantic-ai 1.x but was renamed in 2.x.
from pydantic_ai.result import AgentStream
if not hasattr(AgentStream, "stream_responses"):
    raise RuntimeError(f"This kernel's pydantic-ai is too new for doc_agent.py. Select {PROJECT_DIR / '.venv' / 'bin' / 'python'} as the kernel.")
print("Environment OK")

## 2. `ToolCall` and `collect_tools`

`collect_tools` reads a message history and returns only our real tool calls, in order. Four steps: loop over messages, then parts; skip parts that are not `tool-call`; skip `final_result`; record each remaining name and `args`. It takes the messages as an argument and does not run the agent.

In [ ]:
from dataclasses import dataclass
from typing import Any, Dict


# A small record of one tool call. (@dataclass writes __init__ for us; see the modularization lesson.)
@dataclass
class ToolCall:
    name: str               # Which tool was requested, such as "search".
    args: Dict[str, Any]    # Its arguments. NOTE: really a JSON *string* (Appendix A1).


def collect_tools(messages):
    tool_calls = []
    # Messages contain parts, so we need a loop inside a loop.
    for m in messages:
        for p in m.parts:
            # Ignore prompts, tool returns and text answers. `continue` skips to the next part.
            if p.part_kind != "tool-call":
                continue
            # final_result is the structured-output tool, not one of our tools.
            if p.tool_name == "final_result":
                continue
            tool_calls.append(ToolCall(p.tool_name, p.args))
    return tool_calls

## 3. Free run with a fake model

`FunctionModel` replaces the language model with our own Python function. This script plays the model: **search**, then a **wrong filename**, then the **right** one, then the structured answer. Two invented documents, so **no download, no key, no cost**. It proves the logic, not how a real model behaves.

In [ ]:
import json
from pydantic_ai.models.function import DeltaToolCall, FunctionModel
from doc_agent import DocumentationAgentConfig, create_agent, run_agent_stream, DEFAULT_INSTRUCTIONS
from tools import SearchTools, build_index, create_highlighter, create_documentation_tools_cached

# Two invented documents instead of a download.
fake_docs = [
    {"filename": "judge.md", "title": "LLM as a judge", "description": "Scoring outputs",
     "content": "Use an LLM as a judge to score model outputs against a rubric."},
    {"filename": "bare.md", "content": "A bare file without frontmatter."},
]
fake_tools = SearchTools(build_index(fake_docs), create_highlighter(), {d["filename"]: d["content"] for d in fake_docs})

# A structured answer with every RAGResponse field (models.py).
FAKE_ANSWER = {"answer": "Use an LLM as a judge (judge.md).", "found_answer": True, "confidence": 0.9,
               "confidence_explanation": "Invented.", "answer_type": "how-to", "followup_questions": ["How do I write a rubric?"]}


# Build a model that follows a script of (tool_name, arguments) steps, one per turn.
def make_fake_model(script):
    turn = {"n": 0}

    async def scripted(messages, info):
        name, arguments = script[turn["n"]]
        turn["n"] += 1
        yield {0: DeltaToolCall(name=name, json_args=json.dumps(arguments))}

    return FunctionModel(stream_function=scripted)


SCRIPT = [("search", {"query": "judge"}), ("get_file", {"filename": "wrong.md"}),
          ("get_file", {"filename": "judge.md"}), ("final_result", FAKE_ANSWER)]
agent = create_agent(DocumentationAgentConfig(model=make_fake_model(SCRIPT), instructions=DEFAULT_INSTRUCTIONS), fake_tools)
result = await run_agent_stream(agent, "llm as a judge")   # await: Jupyter already has an event loop
messages = result.new_messages()                          # the recording of this run

Read the output above and the next cell together:

- The `TOOL CALL (search): ...` lines were printed **live** by the callback in `doc_agent.py`. The `(search)` is the **agent's name**, not the tool's.
- The history (below) is a **second view of the same events**, read after the run. Seeing a call in both places does not mean it ran twice.
- The wrong filename still counts as a call: `collect_tools` shows what was *requested*, including mistakes.

In [ ]:
# View 1: every tool-call part, including the structured-output tool.
print("All tool-call parts:", [p.tool_name for m in messages for p in m.parts if p.part_kind == "tool-call"])
# View 2: only our real tools.
tool_calls = collect_tools(messages)
print("collect_tools      :", tool_calls)

# The lesson's checks. Check the length FIRST: indexing a too-short list raises IndexError.
assert len(tool_calls) >= 2
assert tool_calls[0].name == "search"
assert tool_calls[1].name == "get_file"
assert "final_result" not in [c.name for c in tool_calls]
print("Offline tool-call checks passed; no model calls made.")

## 4. The real test (paid, opt-in)

The same checks against the real model. We do not assert how many `get_file` calls happen, since that depends on the question.

**Flaky-test warning:** a real model decides what to do each time. If it answers after one `search`, or searches twice before fetching, `tool_calls[1]` is not `get_file` and the test fails although the code is fine. The test encodes a *hope* about model behavior, not a guarantee.

In [ ]:
# Build tools and an agent exactly like main.py does (cached index if present, else a download).
def create_test_agent():
    tools = create_documentation_tools_cached()
    return create_agent(DocumentationAgentConfig(instructions=DEFAULT_INSTRUCTIONS), tools)


async def test_agent_uses_tools():
    result = await run_agent_stream(create_test_agent(), "llm as a judge")
    tool_calls = collect_tools(result.new_messages())
    print("\nTool calls in order:", [c.name for c in tool_calls])
    assert len(tool_calls) >= 2
    assert tool_calls[0].name == "search"
    assert tool_calls[1].name == "get_file"
    print("test_agent_uses_tools passed")


# Real API calls cost money, so this stays off until you turn it on.
RUN_PAID_TEST = False

if RUN_PAID_TEST:
    assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY (in the environment or documentation-agent/.env) first."
    await test_agent_uses_tools()
else:
    print("Skipped: set RUN_PAID_TEST = True to make real, billed model calls.")

## 5. Project files and pytest

The video keeps `ToolCall` and `collect_tools` in `tests/utils.py` and imports them with `from tests.utils import collect_tools, ToolCall`. This cell creates `tests/__init__.py` and `tests/utils.py` **only if missing**, leaves identical files alone, and **never overwrites** one that differs. It also checks whether `tests/test_agent.py` has the new test (it does not edit it).

In [ ]:
UTILS_SOURCE = '''from dataclasses import dataclass
from typing import Any, Dict


@dataclass
class ToolCall:
    name: str
    args: Dict[str, Any]


def collect_tools(messages):
    tool_calls = []

    for m in messages:
        for p in m.parts:
            part_kind = p.part_kind

            if part_kind != 'tool-call':
                continue

            # final_result is PydanticAI's structured-output tool, not one of our tools
            if p.tool_name == 'final_result':
                continue

            tool_calls.append(ToolCall(p.tool_name, p.args))

    return tool_calls
'''


def ensure_file(path, text):
    if not path.exists():
        path.write_text(text, encoding="utf-8")
        print("Created:", path.relative_to(PROJECT_DIR))
    elif path.read_text(encoding="utf-8") == text:
        print("Unchanged:", path.relative_to(PROJECT_DIR))
    else:
        print("Kept your version (differs from the lesson):", path.relative_to(PROJECT_DIR))


(PROJECT_DIR / "tests").mkdir(exist_ok=True)
ensure_file(PROJECT_DIR / "tests" / "__init__.py", "")
ensure_file(PROJECT_DIR / "tests" / "utils.py", UTILS_SOURCE)

test_file = PROJECT_DIR / "tests" / "test_agent.py"
print("test_agent.py has test_agent_uses_tools:", test_file.exists() and "def test_agent_uses_tools" in test_file.read_text(encoding="utf-8"))

In `tests/test_agent.py` the test is the same body with `@pytest.mark.asyncio` above it (the notebook does not need it, because Jupyter already has an event loop). Run one test from the project folder:

```bash
uv run pytest tests/test_agent.py::test_agent_uses_tools -s
```

`tests/test_agent.py::test_agent_uses_tools` is the **node id** (run only this test). `-s` shows the `TOOL CALL` prints; without it pytest shows output only when a test fails, which is the better default for production (Appendix A2). The cell below only **lists** the tests, so it is free.

In [ ]:
import subprocess

completed = subprocess.run([sys.executable, "-m", "pytest", "--collect-only", "-q", "-p", "no:cacheprovider"],
                           cwd=PROJECT_DIR, capture_output=True, text=True)
print(completed.stdout)
assert completed.returncode == 0, completed.stderr

## Recap, validation, production

- After a run, `result.new_messages()` is a recording of observable steps. A test can read tool calls from it.
- `collect_tools` keeps `tool-call` parts and drops `final_result`. `args` is a JSON string.
- Order assertions: check the length, then index. Real-model tests can be flaky; the fake-model test is deterministic.

**Validated:** executed top to bottom in a scratch copy on `documentation-agent/.venv` (Python 3.13, pydantic-ai 1.59.0, pytest 9.1.1) with `RUN_PAID_TEST = False`. Separately, `uv run pytest tests/test_agent.py -s` passed against the real model, `p.args` was checked as a string, and collection was checked with `tests/__init__.py` removed. **Not run:** this notebook's paid cell, the VS Code test panel, and Python 3.14.2.

### Short production reminder

For production, replace print-based callbacks with structured logs or tracing events. Include request/run IDs, tool name, latency, status, and redacted argument summaries. Do not log sensitive content or full tool arguments by default. Keep callbacks observational; enforce authorization and validation inside the tool execution boundary.

This test shows a tool was *requested*, not that it ran correctly, and it does not measure latency. Before client use: keep paid tests behind a pytest marker, run behavioral tests several times to measure flakiness, assert on answer quality as well as tool order, and run in CI.

# Appendix: tiny examples for the new ideas

Standard library only (A2 also needs pytest in the kernel). No network, no cost. Skip them unless a line confused you.

### A1. `args` is a string, not a dictionary

Type hints such as `Dict[str, Any]` are labels; Python does not enforce them. The real value is JSON text, so parse it when you need the values.

In [ ]:
import json

# Python accepts a string where the hint says Dict[str, Any].
call = ToolCall("search", '{"query": "llm as a judge"}')
print("Stored:", call.args, "| type:", type(call.args).__name__)
# Turn the JSON text into a dictionary to read its values.
print("Parsed:", json.loads(call.args)["query"])

### A2. pytest `-s` and the node id

Writes a tiny test into a temporary folder and runs it with and without `-s`.

In [ ]:
import tempfile

DEMO_TEST = "def test_prints():\n    print('HELLO FROM THE TEST')\n    assert True\n"
with tempfile.TemporaryDirectory() as folder:
    Path(folder, "test_demo.py").write_text(DEMO_TEST, encoding="utf-8")
    for extra in ([], ["-s"]):
        done = subprocess.run([sys.executable, "-m", "pytest", "test_demo.py::test_prints", "-q", "-p", "no:cacheprovider", *extra],
                              cwd=folder, capture_output=True, text=True)
        print(f"flags {extra or '(none)'} -> printed line visible:", "HELLO FROM THE TEST" in done.stdout)

### A3. Order assertions and `IndexError`

A list keeps its order, so `items[0]` is first. Indexing past the end raises `IndexError`, which is why the test checks the length before indexing.

In [ ]:
names = ["search", "get_file"]
assert names[0] == "search" and names[1] == "get_file"
try:
    [][0]
except IndexError as error:
    print("Empty list ->", error)